# Semana 7 — WVC 2026 — Reanálise sem refit (fechamento das tabelas do paper)

**O que este notebook é.** Uma reanálise **das predições já gravadas** no toque único do 7k
(`out_semana56/preds_7k/*.npy`) e dos CSVs das Semanas 2, 3-4 e 5-6. **Não treina, não extrai
feature, não ajusta probe, não abre imagem.** Custo alvo: minutos de CPU.

**Por que existe.** O fechamento experimental deixou seis pendências de *análise* (não de
execução) que mudam o que pode ser escrito:

| # | Pendência | Onde |
|---|---|---|
| 1 | Estatística pareada contra `CLASSICO-MAX` (o clássico mais forte no 7k) — hoje só existe contra `LBP-COR-H` | §A |
| 2 | A tabela pareada existente é de **seed 0**; §1 e H2 são **média sobre seeds**. Em 1% a seed 0 chega a ser o mínimo das cinco | §A |
| 3 | O achado interno-vs-externo (a contribuição real) não tem tabela nem figura | §B |
| 4 | O mecanismo de H2 (custo do PCA por braço) não tem tabela | §C |
| 5 | H4 não tem IC, nem teste de sensibilidade ao comparador; o colapso do `LBP-COR-H` em BACK não está quantificado | §D, §E |
| 6 | `tabelas.tex` omite `CLASSICO-MAX` e `LBP-HF-H`, e a legenda de `tab:h4` não diz entre quais braços é o gap | §G |

---

## Disciplina de toque único `[ler antes de rodar]`

**Reanalisar predições já gravadas NÃO é um novo toque no 7k.** As predições são o produto
sancionado do toque único de 17/08/2026 (pré-commit `df4b2937407f9b87`); nenhum modelo vê o 7k
outra vez aqui.

**Mas nem toda análise nova é confirmatória.** O pré-commit fixou o conjunto de braços, as
condições, as frações, as seeds e a estatística — **e a manchete clássica (`LBP-COR-H`)**. Logo:

- **Família 1 (confirmatória):** comparações contra `LBP-COR-H`. Já estavam especificadas.
- **Família 2 (exploratória, PÓS-HOC):** comparações contra `CLASSICO-MAX`, testes de H4 com
  comparador alternativo, releitura homogêneo-vs-irregular. `CLASSICO-MAX` **estava** no
  pré-commit como braço, mas **não** como comparador manchete — escolhê-lo depois de ver o 7k é
  pós-hoc e tem de ser rotulado assim no paper.

Holm é aplicado **dentro de cada família**, separadamente. Nenhuma família é apagada em função da
outra: a manchete continua sendo a pré-registrada, e a exploratória entra como robustez.

**O que este notebook NÃO conserta:** H3-a continua sendo colapso de pretexto (não há checkpoint
novo); o SimCLR continua subtreinado; a confirmação de Macenko continua documental.

## 0. Configuração

Aponte `OUT_DIR` para a saída da Semana 5-6 (a pasta que contém `preds_7k/`,
`resultados_7k_FINAL.csv`, `precommit_7k.json`). Os únicos dados brutos tocados são os **nomes de
arquivo** do 7k, para reconstruir `y_7k` — nenhuma imagem é aberta.

In [ ]:
# ============================================================================
# 0. CONFIGURACAO (edite so esta celula)
# ============================================================================
from pathlib import Path
import os, json, hashlib, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

OUT_DIR     = Path("./out_semana56")          # <-- EDITE
CRC_VAL_DIR = Path("./CRC-VAL-HE-7K")         # <-- EDITE (so listagem de nomes, nao le imagem)
WEEK56_CSV  = OUT_DIR                          # onde estao os csv da semana 5-6

PRED_DIR = OUT_DIR / "preds_7k"
NEW_DIR  = OUT_DIR / "semana7";  NEW_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = NEW_DIR / "figs";     FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]
NC = len(CLASSES)

# --- herdado, TEM que bater com a Semana 5-6 (senao o bootstrap nao reproduz) --
GLOBAL_SEED  = 20260818
BOOTSTRAP_B  = 2000
COMMON_D     = 32
LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 1.00]
LABEL_SEEDS     = [0, 1, 2, 3, 4]

CLASSICO_MANCHETE = "LBP-COR-H"     # pre-comprometido
CLASSICO_FORTE    = "CLASSICO-MAX"  # POS-HOC: o mais forte medido no 7k
SSL_MANCHETE      = "SimCLR_s0"
FM_REF            = "Phikon"
CLASSICO_BASE     = "LBP"

# H4 pre-registrada (§2 do protocolo). BACK/DEB/LYM NAO foram atribuidas a priori.
H4_MORFO      = ["TUM", "NORM"]
H4_TEXTURAL   = ["STR", "MUS", "ADI", "MUC"]
H4_NAO_PREREG = ["BACK", "DEB", "LYM"]

# --- numeros INTERNOS herdados dos registros anteriores (100% de rotulo) ------
# Fonte: registro_semana3_4.md §1/§5 (Phikon, SimCLR) e h3b_escada_classica_interno.csv (classicos).
# NAO sao recomputados aqui: reabrir o probe interno custaria horas e nao muda nada.
INT100_APRENDIDO = {"Phikon": 99.80, "SimCLR_s0": 97.96}

np.random.seed(GLOBAL_SEED)
print("OUT_DIR:", OUT_DIR.resolve())
print("preds_7k existe:", PRED_DIR.is_dir(),
      "| n arquivos:", len(list(PRED_DIR.glob("*.npy"))) if PRED_DIR.is_dir() else 0)

In [ ]:
# ============================================================================
# 0.1 Carregar artefatos da Semana 5-6
# ============================================================================
def _read(name, **kw):
    p = WEEK56_CSV / name
    if not p.exists():
        print(f"[aviso] {name} nao encontrado em {WEEK56_CSV} - secoes que dependem dele serao puladas.")
        return None
    return pd.read_csv(p, **kw)

df7      = _read("resultados_7k_FINAL.csv")
stat_old = _read("estatistica_pareada_7k.csv")
h3b_int  = _read("h3b_escada_classica_interno.csv")
recall_ext_old = _read("recall_por_classe_7k.csv", index_col=0)
precommit = json.loads((WEEK56_CSV / "precommit_7k.json").read_text()) \
            if (WEEK56_CSV / "precommit_7k.json").exists() else {}

assert df7 is not None, "resultados_7k_FINAL.csv e obrigatorio."
ARMS = list(df7["arm"].unique())
DIMS = df7.groupby("arm")["dim"].first().to_dict()
print("Bracos:", ARMS)
print("Pre-commit hash:", precommit.get("spec_hash"), "| manchete classica:",
      precommit.get("classico_manchete"))
assert precommit.get("classico_manchete", CLASSICO_MANCHETE) == CLASSICO_MANCHETE, \
    "A manchete classica deste notebook diverge do pre-commit. Nao prosseguir sem registrar."

# --- rotulos do 7k: reconstruidos da ORDEM CANONICA de nomes de arquivo ------
Y7_CACHE = NEW_DIR / "y_7k.npy"
if Y7_CACHE.exists():
    y_7k = np.load(Y7_CACHE)
    print("y_7k do cache:", y_7k.shape)
else:
    y = []
    for ci, c in enumerate(CLASSES):
        cdir = CRC_VAL_DIR / c
        if not cdir.is_dir():
            raise FileNotFoundError(f"Subpasta ausente: {cdir}. Aponte CRC_VAL_DIR.")
        n = sum(1 for p in sorted(cdir.iterdir())
                if p.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"})
        y += [ci] * n
    y_7k = np.array(y, dtype=np.int64)
    np.save(Y7_CACHE, y_7k)
    print("y_7k reconstruido e cacheado:", y_7k.shape)

N7 = len(y_7k)
print("7k por classe:", dict(zip(CLASSES, np.bincount(y_7k, minlength=NC))))
assert N7 == 7180, f"N7={N7} != 7180. A pasta do 7k nao e a esperada - PARAR."

## 1. Máquina de bootstrap — idêntica à da Semana 5-6, só que rápida

**Identidade.** As réplicas são geradas com o **mesmo gerador, mesma semente e mesma ordem de
sorteio** (`default_rng(GLOBAL_SEED)`, por classe, na ordem canônica) do notebook original. A
célula 1.2 **verifica** isso reproduzindo linhas da tabela antiga com igualdade quase exata; se
falhar, nada abaixo é comparável e o notebook para.

**Aceleração (sem mudar o resultado).** No bootstrap *estratificado por classe*, o recall da
classe `c` depende apenas do vetor de acertos daquela classe. Então
`balanced_accuracy = média_c( média(acerto_c[idx_c]) )` — não é preciso recomputar matriz de
confusão por réplica. Isso troca ~minutos por ~segundos e é aritmeticamente equivalente.

In [ ]:
# ============================================================================
# 1.1 Indices de bootstrap (reproduzindo a ordem de sorteio da Semana 5-6)
# ============================================================================
POS = [np.where(y_7k == c)[0] for c in range(NC)]
NPC = np.array([len(p) for p in POS])

def build_boot(B, seed):
    """Sorteio na MESMA ordem do notebook original: para cada replica, para cada classe."""
    rng = np.random.default_rng(seed)
    per = [np.empty((B, NPC[c]), dtype=np.int32) for c in range(NC)]
    for b in range(B):
        for c in range(NC):
            per[c][b] = rng.choice(POS[c], size=NPC[c], replace=True)
    return per

BOOT = build_boot(BOOTSTRAP_B, GLOBAL_SEED)
# indice local dentro de cada classe (para indexar o vetor de acertos da classe)
LOCAL = []
for c in range(NC):
    remap = np.full(N7, -1, dtype=np.int32); remap[POS[c]] = np.arange(NPC[c])
    LOCAL.append(remap[BOOT[c]])
print(f"Bootstrap pronto: B={BOOTSTRAP_B}, {NC} classes, n por classe = {NPC.tolist()}")

# ============================================================================
# 1.2 Carregar predicoes -> recall por classe (ponto e bootstrap), com cache
# ============================================================================
def pred_path(arm, cond, frac, seed):
    return PRED_DIR / f"{arm}__{cond}__f{int(frac*100)}__s{seed}.npy"

_CACHE = {}
def recalls(arm, cond, frac, seed):
    """Retorna (recall_ponto[9], recall_boot[B,9]) em %, ou None se a predicao nao existir."""
    key = (arm, cond, frac, seed)
    if key in _CACHE: return _CACHE[key]
    p = pred_path(*key)
    if not p.exists():
        _CACHE[key] = None; return None
    yp = np.load(p)
    ok = (yp == y_7k)
    r_pt = np.empty(NC); r_bt = np.empty((BOOTSTRAP_B, NC))
    for c in range(NC):
        okc = ok[POS[c]]                      # bool: .mean() ja devolve float64
        r_pt[c] = 100.0 * okc.mean()
        r_bt[:, c] = 100.0 * okc[LOCAL[c]].mean(axis=1)
    _CACHE[key] = (r_pt, r_bt)
    return _CACHE[key]

def seeds_for(frac):
    return LABEL_SEEDS if frac < 1.0 else LABEL_SEEDS[:1]

def ba_curves(arm, cond, frac, seeds=None):
    """Balanced accuracy: (ponto_medio_sobre_seeds, boot[B] medio sobre seeds, n_seeds)."""
    seeds = seeds if seeds is not None else seeds_for(frac)
    pts, bts = [], []
    for s in seeds:
        r = recalls(arm, cond, frac, s)
        if r is None: continue
        pts.append(r[0].mean()); bts.append(r[1].mean(axis=1))
    if not pts: return None
    return float(np.mean(pts)), np.mean(bts, axis=0), len(pts)

PREDS_OK = PRED_DIR.is_dir() and len(list(PRED_DIR.glob("*.npy"))) > 0
print("PREDS_OK =", PREDS_OK)

In [ ]:
# ============================================================================
# 1.3 VALIDACAO: as predicoes reproduzem resultados_7k_FINAL.csv e a tabela antiga?
# ============================================================================
if PREDS_OK:
    # (a) alinhamento predicao <-> rotulo: bal_acc recomputada tem que bater com o CSV
    err, checked = 0.0, 0
    for _, row in df7.sample(min(40, len(df7)), random_state=0).iterrows():
        r = recalls(row["arm"], row["cond"], row["frac"], int(row["seed"]))
        if r is None: continue
        err = max(err, abs(r[0].mean() - row["bal_acc"])); checked += 1
    print(f"(a) bal_acc recomputada vs CSV: max|delta| = {err:.2e} em {checked} celulas")
    assert checked > 0 and err < 1e-4, \
        "Predicoes nao batem com resultados_7k_FINAL.csv -> y_7k desalinhado. PARAR."

    # (b) o bootstrap reproduz a tabela pareada antiga (que era seed 0)
    if stat_old is not None:
        rep = []
        for _, row in stat_old.iterrows():
            a = recalls(row["A"], row["cond"], row["frac"], 0)
            b = recalls(row["B"], row["cond"], row["frac"], 0)
            if a is None or b is None: continue
            d = a[1].mean(axis=1) - b[1].mean(axis=1)
            rep.append(dict(A=row["A"], B=row["B"], cond=row["cond"], frac=row["frac"],
                            diff_novo=round(a[0].mean() - b[0].mean(), 2),
                            diff_antigo=row["diff"],
                            lo_novo=round(float(np.percentile(d, 2.5)), 2), lo_antigo=row["ic_lo"],
                            hi_novo=round(float(np.percentile(d, 97.5)), 2), hi_antigo=row["ic_hi"]))
        rep = pd.DataFrame(rep)
        dmax = float(np.abs(rep.diff_novo - rep.diff_antigo).max())
        cimax = float(max(np.abs(rep.lo_novo - rep.lo_antigo).max(),
                          np.abs(rep.hi_novo - rep.hi_antigo).max()))
        print(f"(b) reproducao da tabela antiga: max|delta ponto| = {dmax:.2f} | "
              f"max|delta IC| = {cimax:.2f}")
        assert dmax < 0.011, "Ponto nao reproduz -> convencao divergente. PARAR."
        if cimax > 0.05:
            print("    [ATENCAO] IC difere. Verifique GLOBAL_SEED/BOOTSTRAP_B contra a Semana 5-6.")
        else:
            print("    Bootstrap identico ao da Semana 5-6. Comparacoes novas sao intercambiaveis.")
else:
    print("preds_7k ausente -> secoes A, D, E, F rodam so no que da para fazer com os CSV.")

## A. Estatística pareada — duas convenções, duas famílias

**O problema, em uma frase.** `estatistica_pareada_7k.csv` foi calculada com `seed=0`; as tabelas
§1 e H2 usam média sobre as 5 seeds. Em `frac=1%` isso não é detalhe: para
`SimCLR_s0 − LBP-COR-H` sob PCA32, a seed 0 dá **1,68** e a média das cinco dá **12,12** (range
[1,68 ; 17,95]). O IC bootstrap da seed 0 é estreito e **não é honesto**, porque a fonte de
variância dominante em rótulo baixo — a subamostragem de rótulo — ficou de fora.

**Convenção adotada (registrar no paper):**

| fonte de incerteza | como entra |
|---|---|
| amostragem do conjunto de teste | bootstrap estratificado sobre os 7.180 patches, **B=2000** |
| subamostragem de rótulo | IC95% sobre as 5 seeds (só existe quando `frac < 100%`) |

Para `frac < 100%` o bootstrap é calculado sobre a **predição promediada nas seeds** (para cada
réplica, média das 5 balanced accuracies) — isso ancora o IC no efeito médio, não numa seed
arbitrária. Para `frac = 100%` não há seeds (probe determinístico) e o bootstrap sobre o teste é
o **único** IC honesto disponível. As duas colunas são reportadas lado a lado e **nunca somadas**.

In [ ]:
# ============================================================================
# A. Estatistica pareada estendida
# ============================================================================
FAMILIA_1 = []   # confirmatoria: contra a manchete pre-comprometida
FAMILIA_2 = []   # exploratoria POS-HOC: contra o classico mais forte no 7k

for frac in LABEL_FRACTIONS:
    for cond in ["RAW", f"PCA{COMMON_D}"]:
        FAMILIA_1 += [(SSL_MANCHETE, CLASSICO_MANCHETE, cond, frac),
                      (FM_REF,       CLASSICO_MANCHETE, cond, frac)]
        FAMILIA_2 += [(SSL_MANCHETE, CLASSICO_FORTE, cond, frac),
                      (FM_REF,       CLASSICO_FORTE, cond, frac)]
    FAMILIA_1 += [(FM_REF, SSL_MANCHETE, "RAW", frac),
                  (CLASSICO_MANCHETE, CLASSICO_BASE, "RAW", frac)]
    FAMILIA_2 += [(CLASSICO_FORTE, CLASSICO_BASE, "RAW", frac)]

def holm(p):
    p = np.asarray(p, dtype=float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for k, i in enumerate(order):
        run = max(run, (m - k) * p[i]); adj[i] = min(run, 1.0)
    return adj

def comparar(A, B, cond, frac):
    seeds = seeds_for(frac)
    per_seed = []
    for s in seeds:
        ra, rb = recalls(A, cond, frac, s), recalls(B, cond, frac, s)
        if ra is None or rb is None: return None
        per_seed.append(ra[0].mean() - rb[0].mean())
    per_seed = np.array(per_seed)
    ca, cb = ba_curves(A, cond, frac), ba_curves(B, cond, frac)
    dboot = ca[1] - cb[1]
    lo, hi = np.percentile(dboot, [2.5, 97.5])
    p1 = float((dboot <= 0).mean())
    n = len(per_seed)
    ci_seed = 1.96 * per_seed.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    d_cohen = per_seed.mean() / per_seed.std(ddof=1) if n > 1 and per_seed.std(ddof=1) > 0 else np.nan
    return dict(A=A, B=B, cond=cond, frac=frac, n_seeds=n,
                dif=round(float(per_seed.mean()), 2),
                ic_seed=(round(float(ci_seed), 2) if n > 1 else None),
                boot_lo=round(float(lo), 2), boot_hi=round(float(hi), 2),
                p_boot=round(p1, 4), cohen_d=(round(float(d_cohen), 2) if n > 1 else None),
                dif_seed0=round(float(per_seed[0]), 2))

def rodar_familia(comps, nome):
    rows = [r for r in (comparar(*c) for c in comps) if r is not None]
    if not rows:
        print(f"[{nome}] nenhuma comparacao computavel."); return pd.DataFrame()
    d = pd.DataFrame(rows)
    d["p_holm"] = holm(d["p_boot"].values).round(4)
    d["sig"] = (d.boot_lo > 0) | (d.boot_hi < 0)
    d["familia"] = nome
    return d

if PREDS_OK:
    S1 = rodar_familia(FAMILIA_1, "confirmatoria")
    S2 = rodar_familia(FAMILIA_2, "exploratoria_poshoc")
    STAT = pd.concat([S1, S2], ignore_index=True)
    STAT.to_csv(NEW_DIR / "estatistica_pareada_7k_v2.csv", index=False)
    print("=== FAMILIA 1 - confirmatoria (contra a manchete pre-comprometida) ===")
    print(S1.drop(columns=["familia"]).to_string(index=False))
    print("\n=== FAMILIA 2 - EXPLORATORIA / POS-HOC (contra o classico mais forte no 7k) ===")
    print(S2.drop(columns=["familia"]).to_string(index=False))
    print("\nColunas: dif/ic_seed = media +- IC95% sobre seeds (variancia de rotulo).")
    print("         boot_lo/hi   = IC95% bootstrap sobre o teste (variancia de amostragem).")
    print("         dif_seed0    = o que a tabela ANTIGA reportava. Comparar para ver o vies.")
else:
    STAT = pd.DataFrame()

In [ ]:
# ============================================================================
# A.1 Quanto a convencao antiga (seed 0) enviesava cada linha
# ============================================================================
if PREDS_OK and len(STAT):
    v = STAT[STAT.n_seeds > 1].copy()
    v["vies_seed0"] = (v.dif_seed0 - v.dif).round(2)
    v = v.reindex(v.vies_seed0.abs().sort_values(ascending=False).index)
    print("=== Vies da convencao antiga (dif_seed0 - media 5 seeds), pior primeiro ===")
    print(v[["A", "B", "cond", "frac", "dif", "dif_seed0", "vies_seed0", "ic_seed"]]
          .head(12).to_string(index=False))
    pior = v.iloc[0]
    print(f"\nPior caso: {pior.A} - {pior.B} ({pior.cond}, frac={pior.frac}): "
          f"seed 0 reportava {pior.dif_seed0:+.2f}, a media e {pior.dif:+.2f} "
          f"(vies {pior.vies_seed0:+.2f} pt).")
    print("Algum veredito muda de sinal?",
          bool(((STAT.dif > 0) != (STAT.dif_seed0 > 0)).any()))

## B. Interno vs. externo — a contribuição, com tabela e figura

O gap interno ≈ gap externo para os clássicos **não enriquecidos**, e explode para os
**enriquecidos por cor**. É isto que responde à pergunta do título e é a única parte do trabalho
que **exige** a coorte externa para existir.

Os números internos vêm dos registros anteriores (`h3b_escada_classica_interno.csv` para os
clássicos; §1/§5 do `registro_semana3_4.md` para Phikon e SimCLR). **Eles entram aqui como objeto
de análise — a comparação interno-vs-externo *é* o resultado — e não como números de desempenho.**
A legenda da figura tem de dizer isso.

In [ ]:
# ============================================================================
# B. Amplificacao do gap na troca de coorte
# ============================================================================
ext = df7[df7.cond == "RAW"].groupby(["arm", "frac"]).bal_acc.mean().unstack("frac")

INT100 = dict(INT100_APRENDIDO)
if h3b_int is not None:
    for _, r in h3b_int[h3b_int.frac == 1.0].iterrows():
        INT100[r["arm"]] = float(r["bal_acc"])

CLASSICOS = [a for a in ["Fractal", "Fractal-BM", "LBP", "LBP-H", "LBP-HF-H",
                         "LBP-COR-H", "LBP-ENR", "CLASSICO-MAX"] if a in INT100 and a in ext.index]

rows = []
for cls in CLASSICOS:
    r = dict(classico=cls, d=int(DIMS.get(cls, np.nan)),
             int100=round(INT100[cls], 2), ext100=round(ext.loc[cls, 1.0], 2))
    r["queda"] = round(r["int100"] - r["ext100"], 2)
    for hi, tag in [(FM_REF, "Phi"), (SSL_MANCHETE, "SSL")]:
        if hi in INT100 and hi in ext.index:
            gi = INT100[hi] - INT100[cls]; ge = ext.loc[hi, 1.0] - ext.loc[cls, 1.0]
            r[f"gap{tag}_int"] = round(gi, 2); r[f"gap{tag}_ext"] = round(ge, 2)
            r[f"ampl_{tag}"] = round(ge - gi, 2)
    rows.append(r)
AMPL = pd.DataFrame(rows)
AMPL.to_csv(NEW_DIR / "interno_vs_externo.csv", index=False)
print("=== Gap interno vs externo, 100% de rotulo (RAW) ===")
print(AMPL.to_string(index=False))

enr = AMPL[AMPL.d > 100]; sim = AMPL[AMPL.d <= 100]
print(f"\nAmplificacao media (gap externo - gap interno) vs Phikon:")
print(f"  classicos SIMPLES  (d<=100, n={len(sim)}): {sim.ampl_Phi.mean():+.2f} pt")
print(f"  classicos ENRIQUECIDOS (d>100, n={len(enr)}): {enr.ampl_Phi.mean():+.2f} pt")
print("Leitura: o enriquecimento que fecha o gap DENTRO da coorte e o que nao transfere.")

In [ ]:
# ============================================================================
# B.1 fig5 - dumbbell interno -> externo (a figura que carrega a contribuicao)
# ============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

A = AMPL.sort_values("gapPhi_int")
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ypos = np.arange(len(A))
for i, (_, r) in enumerate(A.iterrows()):
    ax.plot([r.gapPhi_int, r.gapPhi_ext], [i, i], "-", color="0.65", lw=1.6, zorder=1)
ax.scatter(A.gapPhi_int, ypos, s=38, color="#4C72B0", zorder=2, label="internal split (100k)")
ax.scatter(A.gapPhi_ext, ypos, s=38, color="#C44E52", zorder=2, label="external cohort (7k)")
ax.set_yticks(ypos)
ax.set_yticklabels([f"{r.classico} ({r.d}d)" for _, r in A.iterrows()], fontsize=8)
ax.set_xlabel("Phikon $-$ classical, balanced accuracy gap (pts)")
ax.axvline(0, color="0.3", lw=0.8)
ax.legend(fontsize=8, loc="lower right", frameon=True)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_interno_vs_externo.pdf")
print("gravado:", FIG_DIR / "fig5_interno_vs_externo.pdf")
plt.close(fig)

## C. Mecanismo de H2 — o custo do PCA32 por braço

A tabela abaixo é o que transforma "o gap cresce sob PCA" (frágil, dependente do comparador) em
"casar dimensão em `d=32` custa ~1 pt às features aprendidas e 4–15 pts aos descritores clássicos
ricos" (mecânico, verificável, e o mesmo achado de densidade de informação da Semana 2 agora
replicado **fora** do regime saturado).

Inclui o `SimCLR_abl` de propósito: pretexto quebrado ⇒ informação dispersa ⇒ custo alto de PCA,
como um descritor ruim. **Concentração de informação acompanha qualidade de representação, não
dimensão bruta.**

In [ ]:
# ============================================================================
# C. Custo do PCA32 e sensibilidade do veredito de H2 ao comparador
# ============================================================================
M = df7.groupby(["arm", "cond", "frac"]).bal_acc.mean().unstack("frac")
raw, pca = M.xs("RAW", level="cond"), M.xs(f"PCA{COMMON_D}", level="cond")
COST = (raw - pca).round(2)
COST["d"] = [int(DIMS[a]) for a in COST.index]
COST = COST.sort_values("d")
COST.to_csv(NEW_DIR / "custo_pca32.csv")
print(f"=== Custo do PCA{COMMON_D} (RAW - PCA), pontos de balanced accuracy ===")
print(COST.to_string())

apren = [a for a in ["Phikon", "SimCLR_s0", "SimCLR_s1", "SimCLR_s2"] if a in COST.index]
clas_rico = [a for a in CLASSICOS if DIMS[a] > 100]
print(f"\nmedia do custo em 100%: aprendidos {COST.loc[apren, 1.0].mean():.2f} pt | "
      f"classicos ricos (d>100) {COST.loc[clas_rico, 1.0].mean():.2f} pt")

# --- veredito de H2 por comparador -------------------------------------------
rows = []
for hi in [FM_REF, SSL_MANCHETE]:
    for lo in CLASSICOS:
        for f in [0.05, 1.00]:
            gr = raw.loc[hi, f] - raw.loc[lo, f]
            gp = pca.loc[hi, f] - pca.loc[lo, f]
            rows.append(dict(aprendido=hi, classico=lo, d=int(DIMS[lo]), frac=f,
                             gap_raw=round(gr, 2), gap_pca=round(gp, 2), delta=round(gp - gr, 2),
                             veredito=("CRESCE" if gp - gr > 0.5 else
                                       "ENCOLHE" if gp - gr < -0.5 else "PLANO"),
                             paridade_restaurada=bool(gp <= 0)))
H2 = pd.DataFrame(rows)
H2.to_csv(NEW_DIR / "h2_sensibilidade_comparador.csv", index=False)
print("\n=== H2: veredito por comparador classico ===")
print(H2.to_string(index=False))
print(f"\nMenor gap sob PCA{COMMON_D} em qualquer comparacao: {H2.gap_pca.min():+.2f} pt")
print("Paridade restaurada em alguma linha?", bool(H2.paridade_restaurada.any()))
print("=> A afirmacao universal e a FRACA: casar dimensao NUNCA restaura paridade.")
print("   'O gap cresce' e verdade so quando o classico e de alta dimensao - nao generalizar.")

In [ ]:
# ============================================================================
# C.1 fig6 - custo do PCA32 vs dimensao
# ============================================================================
sel = [a for a in COST.index if a in CLASSICOS + apren + ["SimCLR_abl_s0", "SimCLR_abl_s1"]]
fig, ax = plt.subplots(figsize=(6.0, 3.4))
for a in sel:
    if a in CLASSICOS:            col, mk, lab = "#C44E52", "o", "classical"
    elif a.startswith("SimCLR_abl"): col, mk, lab = "#8C8C8C", "^", "SSL (broken pretext)"
    else:                          col, mk, lab = "#4C72B0", "s", "learned"
    ax.scatter(DIMS[a], COST.loc[a, 1.0], color=col, marker=mk, s=45, zorder=3, label=lab)
    ax.annotate(a, (DIMS[a], COST.loc[a, 1.0]), fontsize=6.5,
                xytext=(4, 3), textcoords="offset points")
h, l = ax.get_legend_handles_labels()
seen = dict(zip(l, h)); ax.legend(seen.values(), seen.keys(), fontsize=8, frameon=True)
ax.set_xscale("log"); ax.axhline(0, color="0.3", lw=0.8)
ax.set_xlabel("native feature dimension $d$"); ax.set_ylabel(f"cost of PCA-{COMMON_D} (pts)")
ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig6_custo_pca.pdf"); plt.close(fig)
print("gravado:", FIG_DIR / "fig6_custo_pca.pdf")

## D. Análise de erro por classe — o que realmente derruba o clássico manchete

Três coisas que o texto precisa e que hoje não estão quantificadas:

1. **O colapso do `LBP-COR-H` em BACK.** Sem essa frase, os 79,34 % parecem degradação difusa
   quando são uma falha concentrada numa classe trivial.
2. **Para onde vai o erro** (contrafactual e destino da confusão).
3. **A confusão STR ↔ MUS**, que é o modo de erro dominante de tudo exceto o Phikon.

In [ ]:
# ============================================================================
# D. Recall por classe com IC bootstrap + anatomia do erro
# ============================================================================
ARMS_FOCO = [a for a in [CLASSICO_BASE, CLASSICO_MANCHETE, CLASSICO_FORTE,
                         SSL_MANCHETE, FM_REF] if a in ARMS]

if PREDS_OK:
    rec_pt, rec_ci = {}, {}
    for a in ARMS_FOCO:
        r = recalls(a, "RAW", 1.00, 0)
        if r is None: continue
        rec_pt[a] = r[0]
        rec_ci[a] = np.percentile(r[1], [2.5, 97.5], axis=0)
    REC = pd.DataFrame(rec_pt, index=CLASSES).round(2)
    REC.to_csv(NEW_DIR / "recall_por_classe_7k_v2.csv")
    if recall_ext_old is not None:
        comuns = [a for a in REC.columns if a in recall_ext_old.columns]
        dmax = float((REC[comuns] - recall_ext_old[comuns]).abs().to_numpy().max())
        print(f"[check] recall recomputado vs recall_por_classe_7k.csv: max|delta| = {dmax:.3f} pt")
        assert dmax < 0.02, "Recall divergente do CSV da Semana 5-6 - investigar antes de usar."
    print("=== Recall por classe, 7k, RAW, 100% (IC95% bootstrap entre parenteses) ===")
    show = pd.DataFrame({a: [f"{rec_pt[a][i]:.1f} [{rec_ci[a][0][i]:.1f},{rec_ci[a][1][i]:.1f}]"
                             for i in range(NC)] for a in rec_pt}, index=CLASSES)
    print(show.to_string())

    # --- contrafactual: quanto BACK custa ao classico manchete ---------------
    v = rec_pt[CLASSICO_MANCHETE].copy()
    ib = CLASSES.index("BACK")
    atual = v.mean(); v2 = v.copy(); v2[ib] = 100.0
    print(f"\n{CLASSICO_MANCHETE}: bal.acc = {atual:.2f}. Com BACK em 100%: {v2.mean():.2f} "
          f"(+{v2.mean()-atual:.2f} pt). BACK sozinho custa {v2.mean()-atual:.2f} pt.")
    print(f"  recall BACK: {CLASSICO_MANCHETE} = {v[ib]:.2f}% | "
          + " | ".join(f"{a} = {rec_pt[a][ib]:.2f}%" for a in ARMS_FOCO if a in rec_pt and a != CLASSICO_MANCHETE))

    # --- para onde vai o erro de cada classe ---------------------------------
    def destino(arm, classe, topk=2):
        yp = np.load(pred_path(arm, "RAW", 1.00, 0))
        ci = CLASSES.index(classe); m = (y_7k == ci)
        cnt = np.bincount(yp[m], minlength=NC).astype(float); cnt[ci] = 0
        tot = m.sum()
        idx = np.argsort(-cnt)[:topk]
        return ", ".join(f"{CLASSES[j]} {100*cnt[j]/tot:.1f}%" for j in idx if cnt[j] > 0) or "-"

    print("\n=== Destino do erro (RAW, 100%) ===")
    dest = pd.DataFrame({a: [destino(a, c) for c in CLASSES] for a in ARMS_FOCO}, index=CLASSES)
    print(dest.to_string())

    # --- massa de confusao STR<->MUS ----------------------------------------
    print("\n=== Massa de confusao STR<->MUS (fracao dos patches de STR+MUS trocados) ===")
    for a in ARMS_FOCO:
        yp = np.load(pred_path(a, "RAW", 1.00, 0))
        iS, iM = CLASSES.index("STR"), CLASSES.index("MUS")
        m = np.isin(y_7k, [iS, iM])
        troca = ((y_7k == iS) & (yp == iM)) | ((y_7k == iM) & (yp == iS))
        print(f"  {a:<14} {100*troca.sum()/m.sum():5.2f}%")

## E. H4 com IC e sensibilidade ao comparador

O pré-registro (§2) prevê **gap maior nas morfológicas (TUM, NORM) e menor nas texturais (STR,
MUS, ADI, MUC)**. BACK, DEB e LYM ficam **fora** do teste — não foram atribuídas a priori e não
podem ser realocadas depois de ver o resultado.

Aqui: (i) o teste pré-registrado com IC bootstrap sobre a estatística de grupo, (ii) o mesmo teste
com comparadores alternativos (**pós-hoc**), (iii) a dispersão intra-grupo, que é o que decide se
a média de grupo significa alguma coisa.

In [ ]:
# ============================================================================
# E. H4: teste de grupo com IC + sensibilidade ao comparador
# ============================================================================
if PREDS_OK:
    iM = [CLASSES.index(c) for c in H4_MORFO]
    iT = [CLASSES.index(c) for c in H4_TEXTURAL]

    def h4_teste(A, B):
        ra, rb = recalls(A, "RAW", 1.00, 0), recalls(B, "RAW", 1.00, 0)
        if ra is None or rb is None: return None
        gpt = ra[0] - rb[0]                      # gap por classe (ponto)
        gbt = ra[1] - rb[1]                      # gap por classe (bootstrap)
        est = gpt[iM].mean() - gpt[iT].mean()
        bt = gbt[:, iM].mean(axis=1) - gbt[:, iT].mean(axis=1)
        lo, hi = np.percentile(bt, [2.5, 97.5])
        return dict(aprendido=A, classico=B,
                    morfo=round(float(gpt[iM].mean()), 2),
                    textural=round(float(gpt[iT].mean()), 2),
                    dif_morfo_menos_textural=round(float(est), 2),
                    ic_lo=round(float(lo), 2), ic_hi=round(float(hi), 2),
                    p_boot=round(float((bt <= 0).mean()), 4),
                    amplitude_textural=round(float(gpt[iT].max() - gpt[iT].min()), 2),
                    amplitude_morfo=round(float(gpt[iM].max() - gpt[iM].min()), 2),
                    suporta_H4=bool(lo > 0))

    linhas = []
    for A in [FM_REF, SSL_MANCHETE]:
        for B, tipo in [(CLASSICO_MANCHETE, "pre-registrado"),
                        (CLASSICO_BASE, "pos-hoc"), (CLASSICO_FORTE, "pos-hoc")]:
            r = h4_teste(A, B)
            if r: r["status"] = tipo; linhas.append(r)
    H4 = pd.DataFrame(linhas)
    H4.to_csv(NEW_DIR / "h4_teste_grupo.csv", index=False)
    print("=== H4: morfologicas - texturais, com IC95% bootstrap ===")
    print("(p_boot e UNILATERAL na direcao pre-registrada 'morfologicas > texturais'; "
          "p proximo de 1 significa efeito na direcao OPOSTA. O veredito usa o IC.)")
    print(H4.to_string(index=False))

    pr = H4[H4.status == "pre-registrado"]
    ph = H4[H4.status == "pos-hoc"]
    print(f"\nPre-registrado: dif = {pr.dif_morfo_menos_textural.tolist()} "
          f"(suporta H4: {pr.suporta_H4.tolist()})")
    print(f"Pos-hoc:        dif = {ph.dif_morfo_menos_textural.tolist()} "
          f"(suporta H4: {ph.suporta_H4.tolist()})")
    print("\nAmplitude INTRA-grupo textural vs diferenca ENTRE grupos:")
    for _, r in H4.iterrows():
        print(f"  {r.aprendido} - {r.classico}: entre-grupos {r.dif_morfo_menos_textural:+.2f} pt, "
              f"dentro do grupo textural {r.amplitude_textural:.2f} pt "
              f"=> razao {abs(r.dif_morfo_menos_textural)/max(r.amplitude_textural,1e-9):.2f}")
    print("\nSe a dispersao intra-grupo esmaga a diferenca entre grupos, a media de grupo NAO "
          "sustenta H4 como regra por classe. Reportar as duas coisas.")

In [ ]:
# ============================================================================
# E.1 Gap por classe com IC + releitura POS-HOC (homogeneo vs irregular)
# ============================================================================
if PREDS_OK:
    def gap_classes(A, B):
        ra, rb = recalls(A, "RAW", 1.00, 0), recalls(B, "RAW", 1.00, 0)
        g = ra[0] - rb[0]; gb = ra[1] - rb[1]
        lo, hi = np.percentile(gb, [2.5, 97.5], axis=0)
        return pd.DataFrame({"gap": g.round(2), "ic_lo": lo.round(2), "ic_hi": hi.round(2)},
                            index=CLASSES)

    TIPO = {c: ("morfologica" if c in H4_MORFO else
                "textural" if c in H4_TEXTURAL else "nao-preregistrada") for c in CLASSES}
    G = gap_classes(FM_REF, CLASSICO_MANCHETE)
    G["tipo_H4"] = [TIPO[c] for c in CLASSES]
    G["sig"] = (G.ic_lo > 0) | (G.ic_hi < 0)
    G2 = gap_classes(SSL_MANCHETE, CLASSICO_MANCHETE)
    G["gap_SSL"] = G2.gap; G["ic_lo_SSL"] = G2.ic_lo; G["ic_hi_SSL"] = G2.ic_hi
    G.to_csv(NEW_DIR / "h4_gaps_7k_com_ic.csv")
    print(f"=== Gap por classe vs {CLASSICO_MANCHETE} (7k, RAW, 100%), com IC95% ===")
    print(G.to_string())

    # releitura pos-hoc: homogeneo/quase-periodico vs irregular/multiescala
    HOMOG = ["ADI", "MUS", "LYM"]; IRREG = ["STR", "TUM", "NORM", "MUC", "DEB"]
    for A in [FM_REF, SSL_MANCHETE]:
        g = gap_classes(A, CLASSICO_MANCHETE).gap
        print(f"\n[POS-HOC] {A} - {CLASSICO_MANCHETE}: homogeneo/quase-periodico "
              f"{g[HOMOG].mean():+.2f} pt | irregular/multiescala {g[IRREG].mean():+.2f} pt "
              f"=> separacao {g[IRREG].mean()-g[HOMOG].mean():+.2f} pt")
    print("\n[POS-HOC] Comparar com a separacao morfologica-textural da celula anterior. "
          "Se esta for maior, e a leitura melhor dos dados - mas ROTULAR COMO POS-HOC no paper.")

## F. Robustez: a métrica muda a história? E a não-monotonicidade é real?

Duas checagens que custam segundos e fecham perguntas óbvias de revisor.

1. **F1-macro em vez de balanced accuracy.** Se a ordenação e os sinais dos gaps mudarem, o texto
   precisa saber antes do revisor.
2. **A queda do `LBP-COR-H` de 10% → 100% de rótulo** (84,00 → 79,34) é ruído de seed ou efeito
   real? O bootstrap sobre o teste responde. Se for real, é evidência direta de que **mais rótulo
   faz o probe se apoiar mais nas direções de cor que não transferem** — reforça §B.

In [ ]:
# ============================================================================
# F.1 Robustez de metrica: balanced accuracy vs F1-macro
# ============================================================================
for m in ["bal_acc", "f1_macro"]:
    o = (df7[(df7.cond == "RAW") & (df7.frac == 1.0)]
         .groupby("arm")[m].mean().sort_values(ascending=False))
    print(f"--- ordenacao por {m} (RAW, 100%) ---")
    print("  " + " > ".join(f"{a} {v:.2f}" for a, v in o.items()))
ba = df7[(df7.cond == "RAW") & (df7.frac == 1.0)].groupby("arm").bal_acc.mean()
f1 = df7[(df7.cond == "RAW") & (df7.frac == 1.0)].groupby("arm").f1_macro.mean()
print(f"\ncorrelacao de Spearman entre as duas ordenacoes: "
      f"{ba.rank().corr(f1.rank(), method='spearman'):.4f}")
print("Ordenacao identica?", bool((ba.rank(ascending=False) == f1.rank(ascending=False)).all()))

In [ ]:
# ============================================================================
# F.2 A queda do classico manchete com MAIS rotulo e real?
# ============================================================================
if PREDS_OK:
    def teste_queda(arm, f_alto, f_baixo, cond="RAW"):
        ca, cb = ba_curves(arm, cond, f_alto), ba_curves(arm, cond, f_baixo)
        if ca is None or cb is None: return None
        d = ca[1] - cb[1]; lo, hi = np.percentile(d, [2.5, 97.5])
        return dict(arm=arm, cond=cond, de=f_alto, para=f_baixo,
                    dif=round(ca[0] - cb[0], 2), ic_lo=round(float(lo), 2),
                    ic_hi=round(float(hi), 2), sig=bool(lo > 0 or hi < 0))

    linhas = [teste_queda(a, 0.10, 1.00) for a in
              [CLASSICO_MANCHETE, "LBP-ENR", CLASSICO_FORTE, CLASSICO_BASE, FM_REF, SSL_MANCHETE]
              if a in ARMS]
    Q = pd.DataFrame([l for l in linhas if l])
    Q.to_csv(NEW_DIR / "nao_monotonicidade.csv", index=False)
    print("=== Desempenho em 10% MENOS desempenho em 100% (positivo = mais rotulo PIORA) ===")
    print(Q.to_string(index=False))
    print("\nSe positivo e significativo para os bracos de COR e nulo para LBP/Phikon, a leitura e:")
    print("  mais rotulo => o probe se apoia mais nas direcoes de cor => pior sob troca de coorte.")
    print("  E evidencia direta, dentro do 7k, do mecanismo do achado interno-vs-externo (secao B).")

## G. `tabelas.tex` reconstruído

Conserta:

- **`tab:main`** passa a incluir `CLASSICO-MAX` (o clássico mais forte no 7k, 85,81) e
  `LBP-HF-H`. Omitir o clássico mais forte é a primeira coisa que um comitê de textura nota.
- **`tab:h4`** ganha legenda dizendo **entre quais braços** é o gap, e IC.
- Novas: **`tab:paired`** (estatística pareada, duas famílias, duas fontes de incerteza),
  **`tab:cohort`** (interno vs externo), **`tab:pca`** (custo do PCA).

Ordem das linhas: clássicos por desempenho crescente, depois aprendidos. Tudo em inglês, sem nada
identificável (double-blind).

In [ ]:
# ============================================================================
# G. Gerar tabelas_v2.tex
# ============================================================================
def tex(s):
    return str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

ORDEM_MAIN = [a for a in ["Fractal", "Fractal-BM", "LBP", "LBP-H", "LBP-HF-H", "LBP-COR-H",
                          "LBP-ENR", "CLASSICO-MAX", "SimCLR_s0", "Phikon"] if a in ARMS]
FR = LABEL_FRACTIONS

def celula(arm, cond, frac):
    g = df7[(df7.arm == arm) & (df7.cond == cond) & (df7.frac == frac)].bal_acc.values
    if len(g) == 0: return "--"
    if len(g) == 1: return f"{g[0]:.2f}"
    ci = 1.96 * g.std(ddof=1) / np.sqrt(len(g))
    return f"{g.mean():.2f}\\,$\\pm$\\,{ci:.2f}"

L = []
L.append("% Gerado pelo notebook da Semana 7 (reanalise sem refit). Nao editar a mao.")
L.append("% Fonte: resultados_7k_FINAL.csv, preds_7k/, h3b_escada_classica_interno.csv")
L.append("")

# ---- tab:main ---------------------------------------------------------------
L.append(r"\begin{table}[t]\centering")
L.append(r"\caption{Balanced accuracy (\%) on the external cohort CRC-VAL-HE-7K. Probes are "
         r"trained on stratified subsamples of the NCT-CRC-HE-100K training pool and evaluated on "
         r"the full external cohort. Mean $\pm$ 95\% CI over 5 label seeds; at 100\% the "
         r"subsampling is a no-op and the probe is deterministic (single point). "
         r"$d$ is the native feature dimension.}")
_sslv = np.array([df7[(df7.arm == a) & (df7.cond == "RAW") & (df7.frac == 1.0)].bal_acc.mean()
                  for a in ARMS if a.startswith("SimCLR_s")])
if len(_sslv) > 1:
    L.append(rf"% nota: SD across-pretrain do SimCLR em 100% = {_sslv.std(ddof=1):.2f} pt "
             rf"(n={len(_sslv)} sementes de pre-treino)")
    L[-2] = L[-2][:-1] + (rf" The in-domain SSL arm is shown for one pretraining seed; across "
                          rf"{len(_sslv)} pretraining seeds the 100\% figure has SD "
                          rf"{_sslv.std(ddof=1):.2f}.}}")
L.append(r"\label{tab:main}")
L.append(r"\begin{tabular}{l r rrrrr}\toprule")
L.append(r"Features & $d$ & 1\% & 5\% & 10\% & 25\% & 100\% \\ \midrule")
for i, a in enumerate(ORDEM_MAIN):
    if a == "SimCLR_s0": L.append(r"\midrule")
    L.append(f"{tex(a)} & {int(DIMS[a])} & " +
             " & ".join(celula(a, "RAW", f) for f in FR) + r" \\")
L.append(r"\bottomrule\end{tabular}\end{table}")
L.append("")

# ---- tab:pca ----------------------------------------------------------------
L.append(r"\begin{table}[t]\centering")
L.append(rf"\caption{{Cost of matching the feature dimension to $d={COMMON_D}$: balanced accuracy "
         r"under the raw feature minus balanced accuracy under PCA, on CRC-VAL-HE-7K. "
         r"Dimension matching is nearly free for the learned representations and expensive for the "
         r"high-dimensional classical descriptors; the ablated SSL arm, whose pretext task "
         r"collapsed, behaves like a poor descriptor.}")
L.append(r"\label{tab:pca}")
L.append(r"\begin{tabular}{l r rr}\toprule")
L.append(r"Features & $d$ & 5\% & 100\% \\ \midrule")
for a in [x for x in COST.index if x in ORDEM_MAIN + ["SimCLR_abl_s0"]]:
    L.append(f"{tex(a)} & {int(DIMS[a])} & {COST.loc[a,0.05]:+.2f} & {COST.loc[a,1.0]:+.2f}" + r" \\")
L.append(r"\bottomrule\end{tabular}\end{table}")
L.append("")

# ---- tab:cohort -------------------------------------------------------------
L.append(r"\begin{table}[t]\centering")
L.append(r"\caption{Gap to the frozen foundation model (Phikon $-$ classical, balanced accuracy, "
         r"100\% labels) measured on the internal same-cohort split versus the external cohort. "
         r"For the plain descriptors the two agree; for the colour-enriched ones the gap nearly "
         r"closes internally and reopens externally. Internal figures are reported here as the "
         r"object of this comparison, not as performance results.}")
L.append(r"\label{tab:cohort}")
L.append(r"\begin{tabular}{l r rr r}\toprule")
L.append(r"Classical features & $d$ & internal & external & $\Delta$ \\ \midrule")
for _, r in AMPL.sort_values("gapPhi_int").iterrows():
    L.append(f"{tex(r.classico)} & {int(r.d)} & {r.gapPhi_int:.2f} & {r.gapPhi_ext:.2f} & "
             f"{r.ampl_Phi:+.2f}" + r" \\")
L.append(r"\bottomrule\end{tabular}\end{table}")
L.append("")

# ---- tab:paired -------------------------------------------------------------
if PREDS_OK and len(STAT):
    L.append(r"\begin{table}[t]\centering")
    L.append(r"\caption{Paired differences in balanced accuracy on CRC-VAL-HE-7K. "
             r"Comparisons against the pre-registered classical baseline are confirmatory; those "
             r"against the strongest classical descriptor were selected after the external "
             r"evaluation and are exploratory. Two sources of uncertainty are reported and never "
             r"pooled: the 95\% CI over 5 label seeds (label subsampling), and the 95\% CI from a "
             r"class-stratified bootstrap over the 7{,}180 test patches ($B=2000$; the only "
             r"available interval at 100\% labels, where the probe is deterministic). "
             r"Holm correction is applied within each family.}")
    L.append(r"\label{tab:paired}")
    L.append(r"\begin{tabular}{ll l r rr}\toprule")
    L.append(r"Comparison & Cond. & Labels & $\Delta$ & CI (seeds) & CI (bootstrap) \\ \midrule")
    for fam, lab in [("confirmatoria", "Confirmatory (pre-registered baseline)"),
                     ("exploratoria_poshoc", "Exploratory (strongest classical, post hoc)")]:
        sub = STAT[(STAT.familia == fam) & (STAT.frac.isin([0.01, 0.05, 1.00]))]
        if not len(sub): continue
        L.append(r"\multicolumn{6}{l}{\textit{" + lab + r"}} \\")
        for _, r in sub.iterrows():
            cis = "--" if r.ic_seed is None or (isinstance(r.ic_seed, float) and np.isnan(r.ic_seed)) \
                  else f"$\\pm$\\,{r.ic_seed:.2f}"
            L.append(f"{tex(r.A)} $-$ {tex(r.B)} & {tex(r.cond)} & {int(r.frac*100)}\\% & "
                     f"{r.dif:+.2f} & {cis} & [{r.boot_lo:+.2f}, {r.boot_hi:+.2f}]" + r" \\")
    L.append(r"\bottomrule\end{tabular}\end{table}")
    L.append("")

# ---- tab:h4 -----------------------------------------------------------------
if PREDS_OK:
    L.append(r"\begin{table}[t]\centering")
    L.append(rf"\caption{{Per-class recall gap (percentage points) on CRC-VAL-HE-7K at 100\% "
             rf"labels, between the frozen foundation model and the pre-registered classical "
             rf"baseline ({tex(FM_REF)} $-$ {tex(CLASSICO_MANCHETE)}), with 95\% bootstrap "
             rf"intervals. The morphological/textural grouping was pre-registered; BACK, DEB and "
             rf"LYM were not assigned a priori and are excluded from the directional test.}}")
    L.append(r"\label{tab:h4}")
    L.append(r"\begin{tabular}{l l r l}\toprule")
    L.append(r"Class & Group & Gap (pts) & 95\% CI \\ \midrule")
    nome = {"morfologica": "morphological", "textural": "textural", "nao-preregistrada": "---"}
    for c in CLASSES:
        r = G.loc[c]
        L.append(f"{c} & {nome[r.tipo_H4]} & ${r.gap:+.2f}$ & [{r.ic_lo:+.2f}, {r.ic_hi:+.2f}]" + r" \\")
    L.append(r"\bottomrule\end{tabular}\end{table}")

(NEW_DIR / "tabelas_v2.tex").write_text("\n".join(L), encoding="utf-8")
print("gravado:", NEW_DIR / "tabelas_v2.tex", f"({len(L)} linhas)")
print("\n--- previa ---\n" + "\n".join(L[:22]))

## H. Números para o abstract e para o texto

Emite um bloco único com os valores exatos que vão para o resumo, a introdução e a conclusão —
para não haver arredondamento divergente entre seções. Grava também um **adendo ao registro**,
que é o que se lê ao escrever.

In [ ]:
# ============================================================================
# H. Numeros do abstract + adendo ao registro
# ============================================================================
def g(arm, frac=1.0, cond="RAW"):
    v = df7[(df7.arm == arm) & (df7.cond == cond) & (df7.frac == frac)].bal_acc
    return float(v.mean()) if len(v) else float("nan")

ssl_seeds = [a for a in ARMS if a.startswith("SimCLR_s")]
ssl_v = np.array([g(a) for a in ssl_seeds])

AB = []
def _p(x=""):
    AB.append(str(x)); print(x)

_p("=" * 74)
_p("NUMEROS PARA O ABSTRACT (7k, coorte externa, 100% de rotulo, RAW)")
_p("=" * 74)
_p(f"  Phikon (FM-ref)            {g(FM_REF):.2f}   [ancora publicada eva/kaiko: 93,5]")
_p(f"  SimCLR in-domain (n={len(ssl_v)})     {ssl_v.mean():.2f} +- {ssl_v.std(ddof=1):.2f}  "
      f"(across-pretrain; subtreinado => PISO)")
_p(f"  melhor classico ({CLASSICO_FORTE})  {g(CLASSICO_FORTE):.2f}")
_p(f"  classico manchete ({CLASSICO_MANCHETE})  {g(CLASSICO_MANCHETE):.2f}")
_p(f"  LBP multiescala            {g(CLASSICO_BASE):.2f}")
_p(f"  fractal basico / Bouligand-Minkowski   {g('Fractal'):.2f} / {g('Fractal-BM'):.2f} "
      f"(+{g('Fractal-BM')-g('Fractal'):.2f} pt)")
_p("-" * 74)
_p(f"  vantagem aprendida sobre o MELHOR classico: "
      f"FM {g(FM_REF)-g(CLASSICO_FORTE):+.2f} | SSL {g(SSL_MANCHETE)-g(CLASSICO_FORTE):+.2f}")
_p(f"  vantagem do FM sobre o SSL: {g(FM_REF)-g(SSL_MANCHETE):+.2f} (LIMITE SUPERIOR: SSL subtreinado)")
_p(f"  H2: menor gap sob PCA{COMMON_D} em qualquer comparador = {H2.gap_pca.min():+.2f} pt "
      f"=> paridade NUNCA restaurada")
_p(f"  amplificacao do gap na troca de coorte: classicos simples "
      f"{AMPL[AMPL.d<=100].ampl_Phi.mean():+.2f} pt vs enriquecidos "
      f"{AMPL[AMPL.d>100].ampl_Phi.mean():+.2f} pt")
_p("=" * 74)
(NEW_DIR / "numeros_abstract.txt").write_text("\n".join(AB) + "\n", encoding="utf-8")
print("gravado:", NEW_DIR / "numeros_abstract.txt")

# --- adendo ao registro ------------------------------------------------------
A_ = []
A_.append("# Adendo ao registro final - Semana 7 (reanalise sem refit)\n")
A_.append(f"**Gerado em:** {pd.Timestamp.now():%Y-%m-%d %H:%M}\n")
A_.append(f"**Pre-commit do 7k:** `{precommit.get('spec_hash')}` (inalterado; "
          "nenhuma predicao nova foi gerada).\n")
A_.append("\n## Convencao estatistica adotada\n")
A_.append("Duas fontes de incerteza, reportadas lado a lado e nunca somadas: IC95% sobre 5 seeds "
          "de rotulo (subamostragem) e IC95% bootstrap estratificado sobre os 7.180 patches "
          "(B=2000). Em frac=100% so existe a segunda. A tabela antiga "
          "(`estatistica_pareada_7k.csv`) era de **seed 0** e fica SUPERSEDIDA por "
          "`estatistica_pareada_7k_v2.csv`.\n")
if PREDS_OK and len(STAT):
    v = STAT[STAT.n_seeds > 1].copy(); v["vies"] = (v.dif_seed0 - v.dif).abs()
    A_.append(f"\nVies maximo da convencao antiga: **{v.vies.max():.2f} pt**. "
              f"Nenhum veredito muda de sinal: "
              f"{not bool(((STAT.dif>0)!=(STAT.dif_seed0>0)).any())}.\n")
    A_.append("\n## Estatistica pareada (v2)\n")
    A_.append(STAT[STAT.frac.isin([0.01, 0.05, 1.00])]
              [["familia", "A", "B", "cond", "frac", "dif", "ic_seed", "boot_lo", "boot_hi",
                "p_holm", "sig"]].to_markdown(index=False))
A_.append("\n\n## Interno vs externo (a contribuicao)\n")
A_.append(AMPL.to_markdown(index=False))
A_.append("\n\n## Custo do PCA32 por braco (mecanismo de H2)\n")
A_.append(COST.to_markdown())
A_.append("\n\n## H2: veredito por comparador\n")
A_.append(H2.to_markdown(index=False))
if PREDS_OK:
    A_.append("\n\n## H4: teste de grupo com IC e sensibilidade ao comparador\n")
    A_.append(H4.to_markdown(index=False))
    A_.append("\n\n## Gap por classe com IC (vs manchete)\n")
    A_.append(G.to_markdown())
    A_.append("\n\n## Nao-monotonicidade (10% menos 100%)\n")
    A_.append(Q.to_markdown(index=False))
A_.append("\n\n## Guardas que este notebook NAO remove\n")
A_.append("- H3-a continua sendo colapso de pretexto, nao ablacao de invariancia.\n")
A_.append("- SimCLR continua subtreinado: o gap FM-SSL e limite superior.\n")
A_.append("- Confirmacao de Macenko continua documental, nao empirica.\n")
A_.append("- Comparacoes contra CLASSICO-MAX e a releitura homogeneo-vs-irregular sao "
          "POS-HOC e tem de ser rotuladas assim.\n")
(NEW_DIR / "adendo_semana7.md").write_text("".join(A_), encoding="utf-8")
print("\ngravado:", NEW_DIR / "adendo_semana7.md")
print("Saidas em:", NEW_DIR.resolve())
for p in sorted(NEW_DIR.rglob("*")):
    if p.is_file(): print("   ", p.relative_to(NEW_DIR))